In [74]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd



from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5

from IPython.display import Markdown, display

In [75]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

# Tables and paths

### Paths
ACW_path
PLE_path

### Tables and metrics
#### Block

*ACW*
f"acw_0_subjects_all_{layer_script}.pickle"

f"acw_50_subjects_all_{layer_script}.pickle" 


f"autocorrelation_subjects_all_{layer_script}.pickle"

**variables** =  acw_50_elect_all_epoch_all //  'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all'


*PLE*
f"table_PLE_slope_intercept_subjects_all_{layer_script}.pickle"

f"table_PLE_subjects_all_{layer_script}.pickle"


#### dynamic

f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"

f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle"

**variables** = acw_50_slope_elect_all_epoch_all   // acw_50_std_elect_all_epoch_all 

In [76]:
## Variables of script

##path=analysis_path
path = ACW_path
name_table = f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"


table_df = pd.read_pickle(f"{path}\\{name_table}")


#METRIC NAMES
metric='acw_0_slope_elect_all_epoch_all'


if '_elect_all_epoch_all' in metric:
    print("yes")
    metric_name = metric.replace('_elect_all_epoch_all', "")
else:
    metric_name = metric
    
##CONDITION NAMES

condition_1_name="zinnen"
condition_2_name="woorden"

condition_names = [condition_1_name, condition_2_name]



#TYPE OF DIFFERENCES
difference = "normal"  # "normal" or "inverse"

## this will be used in permutation differences
if difference == "normal":
    alternative = "greater" 
    # this is for cluster permutation test
    tail=1 
    threshold_direction =1
else:
    alternative="less"
    tail=-1
    threshold_direction =-1

number_decimals=6


channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels

##valores de las columnas
condition = table_df["Condition"].unique()
print5("Condiciones en los datos:", condition)

subjects = table_df["Subject"].unique()
print5("Sujetos en los datos:", subjects)

elect_all=  table_df["Elect"].unique()
print5("sensores en los datos:", elect_all)

epochs_all=  table_df["Epoch"].unique()
print5("Epochs en los datos:", epochs_all)
## get info 

#read epochs to build evoked
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subj}_epochs_zinnen_{layer_script}-epo.fif")
len(epochs_zinnen.pick("mag", exclude="bads").ch_names)

evokeds_zinnen=epochs_zinnen.average()
## evokeds_zinnen es una lista

##cojo el primer elemento, solo tengo una lista
evoked_zinnen=evokeds_zinnen

info=evoked_zinnen.info
del epochs_zinnen
del evokeds_zinnen
del evoked_zinnen



yes
['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', '

In [77]:
#lectura de los diccionarios ISC_zinnen y ISC_woorden
dict_woorden_block = pd.read_pickle(ISC_block_path /f"dict_woorden_block.pkl")
dict_zinnen_block = pd.read_pickle(ISC_block_path /f"dict_zinnen_block.pkl")

In [78]:
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted_woorden"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted_zinnen"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=[channels_mag[i] for i in numbers_channels_woorden]
names_channels_zinnen=[channels_mag[i] for i in numbers_channels_zinnen]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

#establecimiento de canales palabras, canales frases y canales mixtos
numbers_channels_only_woorden = list(
    set(numbers_channels_woorden) - set(numbers_channels_zinnen)
)

numbers_channels_only_zinnen = list(
    set(numbers_channels_zinnen) - set(numbers_channels_woorden)
)

number_channels_intersection = list(
    set(numbers_channels_woorden) & set(numbers_channels_zinnen)
)

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)},\
      len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)},\
      len(significant_channels_intersection) {len(number_channels_intersection)}")

#create list of names of channels

names_channels_only_woorden = [channels_mag[i] for i in numbers_channels_only_woorden]
names_channels_only_zinnen = [channels_mag[i] for i in numbers_channels_only_zinnen]
names_channels_intersection = [channels_mag[i] for i in number_channels_intersection]

numbers_channels_woorden [  5  11  15  19  21  22  58  59  68  71  72  74  76  79  83  84  87  88
  89  91  92  93  94 107 118 124 128 131 134 137 139 140 141 142 143 146
 180 184 185 187 188 193 210 215 252 254 258 259 260 261] numbers_channels_zinnen [  5  10  11  12  13  15  18  19  32  36  37  40  41  42  45  46  47  48
  51  60  72  76  78  79  82  83  84  86  87  88  89  90  94  95 118 125
 135 136 139 140 141 142 143 145 173 179 187 188 192 195 197 199 210 211
 212 215 216 217 219 221 222 225 226 227 228 229 230 235 242 243 249 251
 256 269]
len(numbers_channels_woorden) 50 len(numbers_channels_zinnen) 74
names_channels_woorden ['MLC16-4304', 'MLC25-4304', 'MLC42-4304', 'MLC54-4304', 'MLC61-4304', 'MLC62-4304', 'MLO13-4304', 'MLO14-4304', 'MLO41-4304', 'MLO44-4304', 'MLO51-4304', 'MLO53-4304', 'MLP12-4304', 'MLP23-4304', 'MLP34-4304', 'MLP35-4304', 'MLP43-4304', 'MLP44-4304', 'MLP45-4304', 'MLP52-4304', 'MLP53-4304', 'MLP54-4304', 'MLP55-4304', 'MLT25-4304', 'MLT43-4304', 'MLT52

## Posibles comparaciones estadísticas
len(significant_channels_only_woorden): 25, len(significant_channels_only_zinnen): 92, len(significant_channels_intersection) 88

Primer problema, el numero de canales zinnen es claramente superior al de woorden, big problem, aunque la interseccion es alta, tal como esperaríamos. Seguramente esto sea porque hay ruido 

- comparación ACW-50 en zinnen entre estos canales zinnen y canales woorden  + comapracion en woorden de lo mismo

- comparación entre solo zinnen con intersection  en woorden y en zinnen (aunque ojo, esto es básicamente aceptar que los canales solo Zinnen son ruido)

- comapración en el promedio general de acw en ambas condiciones de los canalaes zinnen vs promedio general de canales woorden

- comparación de la intersección vs canales no signficativos: no sé muy bien como interpretar esto

- 

In [79]:
#filtras por sujeto 
channels_type=["ch_only_woorden", "ch_only_zinnen", "ch_intersection"]

dict_metric_condition_subj_epoch_mean_ch_type = {}
dict_metric_condition_subj_epoch_mean_ch_type_mean = {}


for cond in condition_names:
    for ch_type in channels_type:
        # for subj in subjects:
            #creacion de lista de valores de metric para cada sujeto
            
        #eliges la condicion y el tipo de canal y te quedas con los valores 
        if ch_type == "ch_only_woorden":
            elects = names_channels_only_woorden
        elif ch_type == "ch_only_zinnen":
            elects = names_channels_only_zinnen
        elif ch_type == "ch_intersection":
            elects = names_channels_intersection
        
        array_metric_condition_subj_epoch_mean_ch_type = (
            table_df
            .query("Elect in @elects")
            .query("Condition == @cond")
            .groupby(["Subject", "Elect"])[metric]
            .mean().reset_index()
            .pivot(index="Subject", columns="Elect", values=metric)
            .to_numpy()
        )
        
        array_metric_condition_subj_epoch_mean_ch_type_mean = (
            table_df
            .query("Elect in @elects")
            .query("Condition == @cond")
            .groupby("Subject")[metric]
            .mean()
            .values
        )
        
                
        
        dict_metric_condition_subj_epoch_mean_ch_type[f"X_{cond}_{ch_type}"] = array_metric_condition_subj_epoch_mean_ch_type
        dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{cond}_{ch_type}"] = array_metric_condition_subj_epoch_mean_ch_type_mean


# 1. Comparaciones ch_Zinnen only vs ch_intersection
These are gonne be comparisons between channels zinnen vs intersection in each condition.
E.g. in zinnen condition, we compare acw in 

In [80]:

from scipy.stats import permutation_test

##dependent condition
def paired_statistic(diff, _):
    return np.mean(diff)


# Independent condition
def diff_means(x, y):
    return np.mean(x) - np.mean(y)

experimental_condition=["zinnen", "woorden"]
type_channel=["intersection", "only_woorden"]

## 1.1 Comparisons in each experimental condition

In [81]:
if difference == "normal":     
    display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**")) 
else:     
    display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))


# # if difference == "normal":
# #     print(f"Difference is normal, alternative hypothesis is zinnen > woorden")
# # else:
# #     print(f"Difference is inverse, alternative hypothesis is less (for PLE condition) alternative hypothesis is zinnen < woorden")

dependency= ["Dependent", "Independent"]
for type_dependency in dependency:
    print("------------------------------------------------------------------------------")
    print("------------------------------------------------------------------------------")

    print(f"{type_dependency.upper()} analysis in {metric_name}")

    for exp in experimental_condition:
        print("------------------------------------------------------------------------------")
        print(f"in {exp} condition")
        print("------------------------------------------------------------------------------")
        for ch in type_channel:

            print(f"for differences between only_zinnen  and {ch}:")

            # print(f"Processing {exp} for differences between zinenn and {ch}...")
            ##notice that in this comparison X is alwas ch_zinnnen_mean
            x= dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{exp}_ch_only_zinnen"]
            ##notice that in n this comparison y is  ch_woorden_mean or ch_intersection_mean
            y=dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{exp}_ch_{ch}"]


            diff=x-y
            
            if type_dependency ==  "Dependent":
                # Permutation test
                res = permutation_test(
                    (diff, np.zeros_like(diff)),
                    #
                    statistic=paired_statistic,
                    vectorized=False,
                    n_resamples=10000,
                    alternative=alternative, 
                    random_state=42
                )

                # Cohen's d
                mean_diff = np.mean(diff)
                std_diff = np.std(diff, ddof=1)
                cohens_d = mean_diff / std_diff
            
            if type_dependency == "Independent":
                
                res = permutation_test(
                    (x, y),
                    statistic=diff_means,
                    vectorized=False,
                    n_resamples=10000,
                    alternative=alternative, 
                    random_state=42
                )

                # Calcular Cohen's d (independent)
                mean_x, mean_y = np.mean(x), np.mean(y)
                std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
                n_x, n_y = len(x), len(y)

                pooled_std = np.sqrt(((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2))

                if difference == "normal":
                    cohens_d = (mean_x - mean_y) / pooled_std
                else:
                    cohens_d = (mean_y - mean_x) / pooled_std


            print(f"   Statistic value: {res.statistic:.{number_decimals}f}")
            print(f"   p-value        : {res.pvalue:.{number_decimals}f}")
            print(f"   Cohen's d      : {cohens_d:.{number_decimals}f}")

**Difference is normal**, alternative hypothesis is **zinnen > woorden**

------------------------------------------------------------------------------
------------------------------------------------------------------------------
DEPENDENT analysis in acw_0_slope
------------------------------------------------------------------------------
in zinnen condition
------------------------------------------------------------------------------
for differences between only_zinnen  and intersection:
   Statistic value: 0.000240
   p-value        : 0.059194
   Cohen's d      : 0.387827
for differences between only_zinnen  and only_woorden:
   Statistic value: 0.000220
   p-value        : 0.123488
   Cohen's d      : 0.288637
------------------------------------------------------------------------------
in woorden condition
------------------------------------------------------------------------------
for differences between only_zinnen  and intersection:
   Statistic value: -0.000209
   p-value        : 0.884712
   Cohen's d      : -0.292336
for differences between

## 1.2 Using average in both conditions


In [82]:
if difference == "normal":
    display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**"))
else:
    display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))

# # if difference == "normal":
# #     print(f"Difference is normal, alternative hypothesis is zinnen > woorden")
# # else:
# #     print(f"Difference is inverse, alternative hypothesis is less (for PLE condition) alternative hypothesis is zinnen < woorden")

dependency = ["Dependent", "Independent"]

for type_dependency in dependency:
    print("------------------------------------------------------------------------------")
    print("------------------------------------------------------------------------------")
    print(f"{type_dependency.upper()} analysis in {metric_name}")

    for ch in type_channel:
        print("------------------------------------------------------------------------------")
        print(f"for differences between only_zinnen and {ch}:")
        print("------------------------------------------------------------------------------")
        # Promediar condiciones para X y Y
        x = (
            dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_only_zinnen"]
            + dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_only_zinnen"]
        ) / 2

        y = (
            dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_{ch}"]
            + dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_{ch}"]
        ) / 2

        # Calcular diferencia
        diff = x - y

        if type_dependency == "Dependent":
            res = permutation_test(
                (diff, np.zeros_like(diff)),
                statistic=paired_statistic,
                vectorized=False,
                n_resamples=10000,
                alternative=alternative,
                random_state=42
            )

            # Cohen's d (dependiente)
            mean_diff = np.mean(diff)
            std_diff = np.std(diff, ddof=1)
            cohens_d = mean_diff / std_diff

        elif type_dependency == "Independent":
            res = permutation_test(
                (x, y),
                statistic=diff_means,
                vectorized=False,
                n_resamples=10000,
                alternative=alternative,
                random_state=42
            )

            # Cohen's d (independiente)
            mean_x, mean_y = np.mean(x), np.mean(y)
            std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
            n_x, n_y = len(x), len(y)

            pooled_std = np.sqrt(
                ((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2)
            )

            if difference == "normal":
                cohens_d = (mean_x - mean_y) / pooled_std
            else:
                cohens_d = (mean_y - mean_x) / pooled_std

        # Imprimir resultados
        print(f"   Statistic value: {res.statistic:.{number_decimals}f}")
        print(f"   p-value        : {res.pvalue:.{number_decimals}f}")
        print(f"   Cohen's d      : {cohens_d:.{number_decimals}f}")

**Difference is normal**, alternative hypothesis is **zinnen > woorden**

------------------------------------------------------------------------------
------------------------------------------------------------------------------
DEPENDENT analysis in acw_0_slope
------------------------------------------------------------------------------
for differences between only_zinnen and intersection:
------------------------------------------------------------------------------
   Statistic value: 0.000015
   p-value        : 0.460854
   Cohen's d      : 0.024857
------------------------------------------------------------------------------
for differences between only_zinnen and only_woorden:
------------------------------------------------------------------------------
   Statistic value: 0.000206
   p-value        : 0.127487
   Cohen's d      : 0.279940
------------------------------------------------------------------------------
------------------------------------------------------------------------------
INDEPENDENT analysis in acw_0_slope
----------------

# 2. Comparison of each channel type in both conditions

Before we compared differences of word and sentence channels. Now we are going to compare the channels with themselves in different experimental conditions

In [83]:


if difference == "normal":
    display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**"))
else:
    display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))


# here i have to take the list of ALL types of channels, not like before, where we always take only_zinnen
all_type_channels = ['only_zinnen','intersection', 'only_woorden']

for ch in all_type_channels:
    print("------------------------------------------------------------------------------")
    print(f"for differences in {metric_name} ch {ch}  between zinnen and word condition:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")

    #ch value in zinnen condition
    x= dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_{ch}"]
    y= dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_{ch}"]

    print(f"X is: X_{experimental_condition[0]}_ch_{ch}", x.shape)
    print(f"Y is: X_{experimental_condition[1]}_ch_{ch}", y.shape)


    if difference=="normal":
        diff=x-y
    elif difference=="inverse":
        diff=y -x
        # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        statistic=paired_statistic,
        vectorized=False,
        n_resamples=10000,
        alternative=alternative, 
        random_state=42
    )

    # Calcular Cohen's d para muestras emparejadas
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    
    if difference == "normal":
        cohens_d = (mean_x - mean_y) / pooled_std
    else:
        cohens_d = (mean_y - mean_x) / pooled_std
        
    # Imprimir resultados
    print(f"   Statistic value: {res.statistic:.{number_decimals}f}")
    print(f"   p-value        : {res.pvalue:.{number_decimals}f}")
    print(f"   Cohen's d      : {cohens_d:.{number_decimals}f}")

**Difference is normal**, alternative hypothesis is **zinnen > woorden**

------------------------------------------------------------------------------
for differences in acw_0_slope ch only_zinnen  between zinnen and word condition:
X is: X_zinnen_ch_only_zinnen (18,)
Y is: X_woorden_ch_only_zinnen (18,)
   Statistic value: 0.000322
   p-value        : 0.048195
   Cohen's d      : 0.245351
------------------------------------------------------------------------------
for differences in acw_0_slope ch intersection  between zinnen and word condition:
X is: X_zinnen_ch_intersection (18,)
Y is: X_woorden_ch_intersection (18,)
   Statistic value: -0.000127
   p-value        : 0.739026
   Cohen's d      : 0.245351
------------------------------------------------------------------------------
for differences in acw_0_slope ch only_woorden  between zinnen and word condition:
X is: X_zinnen_ch_only_woorden (18,)
Y is: X_woorden_ch_only_woorden (18,)
   Statistic value: 0.000294
   p-value        : 0.069393
   Cohen's d      : 0.245351
